# Garbage Classification – Improved Teaching CNN

This notebook is an improved version of the baseline `GarbageClassification_CNN.ipynb`.

It follows a structured teaching progression:
1. Switch from grayscale to **RGB** input
2. Increase model capacity (Conv-BN-ReLU blocks)
3. Add **BatchNormalization** for stable training
4. Add **data augmentation** to improve generalization
5. Use **ReduceLROnPlateau** and **EarlyStopping** callbacks
6. Visualize training history (accuracy and loss over epochs)
7. Export an int8 `.tflite` for edge deployment

> **Key concept:** Before tuning optimizers or activations, compare train vs. val accuracy.
> - Both low → model is **underfitting** (not enough capacity)
> - Train high, val low → model is **overfitting** (too much capacity or too little data)


In [ ]:
!pip install kagglehub==0.3.5


In [ ]:
import os
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = ""

np.random.seed(0)
tf.random.set_seed(0)

print('TensorFlow:', tf.__version__)


## 1) Dataset

Same dataset options as the baseline notebook (KaggleHub or local folder).

**Key change:** Images are now loaded as **RGB (3-channel)** instead of grayscale.
Color information helps distinguish visually similar categories like cardboard vs. paper and glass vs. plastic.


In [ ]:
USE_KAGGLEHUB = True
KAGGLE_DATASET = "asdasdasasdas/garbage-classification"
FORCE_CLASS_ROOT = None

DATA_DIR = Path('data')
IMG_SZ = 96
BATCH = 64
VAL_FRAC = 0.2

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


def _has_images(p: Path) -> bool:
    for f in p.iterdir():
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            return True
    return False


def find_class_root(root: Path, max_depth: int = 6) -> Path:
    """Find a directory whose immediate subfolders look like class folders."""
    root = root.resolve()
    best = None
    best_score = (-1, -1)

    def depth(p: Path) -> int:
        try:
            return len(p.relative_to(root).parts)
        except Exception:
            return 999

    dirs = [root]
    for p in root.rglob('*'):
        if p.is_dir() and depth(p) <= max_depth:
            dirs.append(p)

    for d in dirs:
        subdirs = [s for s in d.iterdir() if s.is_dir()]
        if len(subdirs) < 2:
            continue
        class_dirs = [s for s in subdirs if _has_images(s)]
        if len(class_dirs) < 2:
            continue

        img_count = 0
        for c in class_dirs:
            img_count += sum(1 for f in c.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS)
        score = (len(class_dirs), img_count)
        if score > best_score:
            best_score = score
            best = d

    if best is None:
        raise RuntimeError(f"Could not find class folders under: {root}")
    return best


def index_from_class_root(class_root: Path):
    class_dirs = [d for d in class_root.iterdir() if d.is_dir() and _has_images(d)]
    class_names = sorted([d.name for d in class_dirs])
    name_to_id = {n: i for i, n in enumerate(class_names)}

    items = []
    for cname in class_names:
        cdir = class_root / cname
        for p in sorted(cdir.iterdir()):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                items.append((str(p), name_to_id[cname]))
    return items, class_names


def split_items(items, val_frac=0.2):
    from collections import defaultdict

    by_cls = defaultdict(list)
    for p, y in items:
        by_cls[int(y)].append(p)

    train, val = [], []
    rng = np.random.default_rng(0)
    for y, paths in by_cls.items():
        paths = list(paths)
        rng.shuffle(paths)
        n_val = int(len(paths) * val_frac)
        val_paths = paths[:n_val]
        train_paths = paths[n_val:]
        train.extend([(p, y) for p in train_paths])
        val.extend([(p, y) for p in val_paths])

    rng.shuffle(train)
    rng.shuffle(val)
    return train, val


def load_image_label(path, y):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)  # RGB
    img = tf.image.resize_with_pad(img, IMG_SZ, IMG_SZ)
    img = tf.cast(tf.clip_by_value(img, 0, 255), tf.float32) / 255.0
    return img, tf.cast(y, tf.int32)


def ds_from_items(items, shuffle=False):
    paths = [p for p, _ in items]
    labels = [int(y) for _, y in items]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(4096, len(paths)), seed=0)
    ds = ds.map(load_image_label, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds


train_dir = DATA_DIR / 'train'
val_dir = DATA_DIR / 'val'

if (not USE_KAGGLEHUB) and train_dir.exists() and val_dir.exists():
    print('Using folder dataset:', train_dir, val_dir)
    train_ds = tf.keras.utils.image_dataset_from_directory(
        str(train_dir),
        labels='inferred',
        label_mode='int',
        color_mode='rgb',
        image_size=(IMG_SZ, IMG_SZ),
        batch_size=BATCH,
        shuffle=True,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        str(val_dir),
        labels='inferred',
        label_mode='int',
        color_mode='rgb',
        image_size=(IMG_SZ, IMG_SZ),
        batch_size=BATCH,
        shuffle=False,
    )
    class_names = train_ds.class_names
    train_ds = train_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
else:
    import kagglehub
    path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print('KaggleHub dataset path:', path)

    class_root = Path(FORCE_CLASS_ROOT) if FORCE_CLASS_ROOT else find_class_root(path)
    print('Class root:', class_root)

    items, class_names = index_from_class_root(class_root)
    train_items, val_items = split_items(items, val_frac=VAL_FRAC)
    print('Train images:', len(train_items), 'Val images:', len(val_items))
    print('Classes:', class_names)

    train_ds = ds_from_items(train_items, shuffle=True)
    val_ds = ds_from_items(val_items, shuffle=False)


NUM_CLASSES = len(class_names)
print('NUM_CLASSES:', NUM_CLASSES)


In [ ]:
# Check class balance
from collections import Counter

if 'train_items' in dir():
    train_counts = Counter([y for _, y in train_items])
    val_counts = Counter([y for _, y in val_items])
    print("Train class distribution:")
    for i, name in enumerate(class_names):
        print(f"  {name}: {train_counts[i]}")
    print("\nVal class distribution:")
    for i, name in enumerate(class_names):
        print(f"  {name}: {val_counts[i]}")


In [ ]:
# Visualize a small batch (RGB)
x, y = next(iter(train_ds))
plt.figure(figsize=(10, 5))
for i in range(min(8, x.shape[0])):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x[i].numpy())
    plt.title(class_names[int(y[i])])
    plt.axis('off')
plt.suptitle('Sample Training Images (RGB)', fontsize=13)
plt.tight_layout()
plt.show()


## 2) Improved Teaching CNN

### Key changes from the baseline
- **RGB input** (3 channels instead of 1)
- **Increased capacity**: ~300k+ parameters instead of ~2k
- **Conv → BatchNorm → ReLU** blocks for stable training
- **Dropout** before the final dense layer for regularization
- **Data augmentation** applied inline at training time

### BatchNormalization explained
Batch normalization normalizes layer inputs across a mini-batch so they have mean ≈ 0 and variance ≈ 1, then applies learned scale (γ) and shift (β) parameters:

```
y_norm = (y - μ) / √(σ² + ε)
y_bn = γ · y_norm + β
```

**Why it helps:**
- Reduces internal covariate shift
- Makes gradient descent more stable
- Often allows higher learning rates
- Acts as a mild regularizer


In [ ]:
from tensorflow.keras import layers, models


# ── Data augmentation (applied only during training) ──────────────────────────
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')


# ── Conv-BN-ReLU helper ───────────────────────────────────────────────────────
def conv_bn_relu(x, filters, kernel_size=3):
    x = layers.Conv2D(filters, kernel_size, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


# ── Build model ───────────────────────────────────────────────────────────────
def build_teaching_cnn(input_shape=(96, 96, 3), num_classes=6):
    inp = layers.Input(shape=input_shape)

    # Augmentation (no-op at inference time)
    x = augment(inp)

    # Block 1
    x = conv_bn_relu(x, 32)
    x = conv_bn_relu(x, 32)
    x = layers.MaxPooling2D()(x)

    # Block 2
    x = conv_bn_relu(x, 64)
    x = conv_bn_relu(x, 64)
    x = layers.MaxPooling2D()(x)

    # Block 3
    x = conv_bn_relu(x, 128)
    x = layers.MaxPooling2D()(x)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inp, out, name='teaching_cnn')


model = build_teaching_cnn(input_shape=(IMG_SZ, IMG_SZ, 3), num_classes=NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()


## 3) Train with Learning-Rate Callbacks

Two callbacks are added:
- **`ReduceLROnPlateau`**: halves the learning rate when val_loss stops improving for 5 epochs (min lr = 1e-5)
- **`EarlyStopping`**: stops training early and restores the best weights if val_loss does not improve for 12 epochs

> **Lesson:** Learning rate scheduling is often more impactful than switching optimizer types.


In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks,
    verbose=2,
)


## 4) Training History

Visualize model accuracy and loss over each epoch for both training and validation sets.

- If **both** curves are low → underfitting (increase capacity)
- If **train is high** but **val is low** → overfitting (add regularization / augmentation)


In [ ]:
# Plot training history: Accuracy and Loss over epochs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy ---
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
axes[0].set_title('Model Accuracy over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Loss ---
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
axes[1].set_title('Model Loss over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5) Confusion Matrix

Overall accuracy can hide class-specific failure. Inspecting the confusion matrix reveals which classes the model confuses most often.


In [ ]:
y_true = []
y_pred = []

for xb, yb in val_ds:
    preds = model.predict(xb, verbose=0).argmax(axis=1)
    y_true.extend(yb.numpy())
    y_pred.extend(preds)

cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES).numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Validation Confusion Matrix')

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()


## 6) Export int8 `.tflite` for Edge Deployment

After training a capable model, we compress it back to int8 quantization for edge deployment (e.g., OpenMV).

> **Lesson:** First prove the dataset is learnable with a larger model. Then compress for deployment and observe how accuracy changes — this is the accuracy/latency/model-size tradeoff.


In [ ]:
def representative_dataset(ds, num_batches=50):
    for x, _ in ds.take(num_batches):
        for i in range(x.shape[0]):
            yield [tf.expand_dims(x[i], 0)]


os.makedirs('models', exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_dataset(train_ds)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite = converter.convert()
tflite_path = os.path.join('models', 'teaching_cnn_int8.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite)

labels_path = os.path.join('models', 'teaching_cnn_labels.txt')
with open(labels_path, 'w', encoding='utf-8') as f:
    for n in class_names:
        f.write(n + '\n')

print('Wrote', tflite_path, 'bytes=', os.path.getsize(tflite_path))
print('Wrote', labels_path)
print('Class names:', class_names)


## 7) Sanity-check the Exported Model

Verify that the exported `.tflite` model runs correctly before deploying to hardware.


In [ ]:
import numpy as np

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
inp_details = interpreter.get_input_details()[0]
out_details = interpreter.get_output_details()[0]
print('Input:', inp_details)
print('Output:', out_details)

xb, yb = next(iter(val_ds))
x0 = xb[0:1].numpy()
y0 = int(yb[0].numpy())

scale, zp = inp_details['quantization']
xi = np.round(x0 / scale + zp).astype(np.int8)

interpreter.set_tensor(inp_details['index'], xi)
interpreter.invoke()
yo = interpreter.get_tensor(out_details['index'])[0]

oscale, ozp = out_details['quantization']
probs = (yo.astype(np.float32) - ozp) * oscale
pred = int(np.argmax(probs))
print(f'True: {class_names[y0]}  Pred: {class_names[pred]}')
